# AgentCore Long Memory Example

This notebook demonstrates how to use Bedrock AgentCore Memory with LangGraph to create a nutrition assistant that remembers user preferences across conversations.


In [31]:
import os
import logging
import uuid

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph_checkpoint_aws import AgentCoreMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt


## Configuration

Set up the region, logging, memory name, and API keys.


In [ ]:
region = os.getenv('AWS_REGION', 'us-east-1')
logging.getLogger("math-agent").setLevel(logging.DEBUG)

memory_name = "NutritionAssistant"
BEDROCK_MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"



# Get execution role ARN from environment variable or use default
memory_execution_role_arn = os.getenv(
    'MEMORY_EXECUTION_ROLE_ARN',
    'arn:aws:iam::YOUR_ACCOUNT:role/BedrockAgentCoreExecutionRole'  # Default role name
)


## Initialize Memory Client

Create the memory client and set up the memory with custom strategies for capturing nutrition preferences.


In [33]:
client = MemoryClient(region_name=region)

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn=memory_execution_role_arn,
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaces": ["/{actorId}/preferences"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": BEDROCK_MODEL_ID,  # Must be a Bedrock model ID
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": BEDROCK_MODEL_ID,  # Must be a Bedrock model ID
                        }
                    }
                }
            }
        },
    ]
)
memory_id = memory["id"]
print(f"Memory created/retrieved with ID: {memory_id}")


Failed to create memory: An error occurred (ValidationException) when calling the CreateMemory operation: Validation failed during CreateMemory: Memory with name NutritionAssistant already exists


Memory created/retrieved with ID: NutritionAssistant-biR2bnH4Ve


## Initialize Store and LLM

Set up the AgentCore Memory Store for long-term memory and initialize the LLM.


In [34]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize OpenAI LLM for the main conversation
# Note: The memory operations use Bedrock (configured above), but the main LLM can use OpenAI
llm = init_chat_model(BEDROCK_MODEL_ID, model_provider="bedrock_converse", region_name=region)

# Note: We'll use client.create_event() directly in hooks to trigger extraction


## Define Hooks

Create pre-model and post-model hooks to save and retrieve messages from AgentCore Memory.


In [35]:

def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    print("pre_model_hook")
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)
    
    messages = state.get("messages", [])
    print(f"Pre model hook messages: {messages}")
    # Save the last human message we see before LLM invocation
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    # Retrieve user preferences based on the last message and append to state
    user_preferences_namespace = (actor_id, "preferences")
    preferences = store.search(user_preferences_namespace, query=msg.content, limit=5)
    
    # Construct another AI message to add context before the current message
    if preferences:
        context_items = [pref.value for pref in preferences]
        context_message = AIMessage(
            content=f"[User Context: {', '.join(str(item) for item in context_items)}]"
        )
        # Insert the context message before the last human message
        return {"messages": messages[:-1] + [context_message, messages[-1]]}
    
    return {"llm_input_messages": messages}




In [36]:

def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs post-LLM invocation to save the latest human message"""
    print("post_model_hook")
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)
    
    messages = state.get("messages", [])
    print(f"Post model hook messages: {messages}")
    # Save the LLMs response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    
    return {"messages": messages}


## Create the Agent Graph

Create the LangGraph agent with the memory store and hooks.


In [37]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[], # No additional tools needed for this example
    checkpointer=InMemorySaver(), # For conversation state management
    pre_model_hook=pre_model_hook, # Retrieves user preferences before LLM call
    post_model_hook=post_model_hook  # Saves conversation after LLM response
)


/var/folders/q_/4ck_g4y11cv5cb5y9jhslxsm0000gn/T/ipykernel_16629/3805340840.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(


## Set Up Configuration

Configure the actor ID and thread ID for the conversation.


In [38]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1", # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id, # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}


## Helper Function

Create a helper function to run the agent and pretty print the output.


In [39]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


## Example 1: First Conversation

Run the first conversation about cooking salmon.


In [40]:
prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)


================================ Human Message =================================


Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?

pre_model_hook
Pre model hook messages: [HumanMessage(content='\nHey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has\ngreat macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better\nand also improve the protein and vitamins I get?\n', additional_kwargs={}, response_metadata={}, id='69185236-460e-4e59-b58c-df19d166c75c')]


/Users/arnab/projects/llm/agentcore-long-memory/.venv/lib/python3.12/site-packages/langgraph_checkpoint_aws/agentcore/store.py:125: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  event_messages = convert_langchain_messages_to_event_messages([message])


================================== Ai Message ==================================

[User Context: {'content': '{"context":"The user mentions that salmon with rice and veggies is one of their favorite meals.","preference":"Enjoys salmon with rice and vegetables","categories":["food","meals","health"]}', 'memory_strategy_id': 'NutritionPreferences-mhcrH7Dq74', 'namespaces': ['/user-1/preferences']}, {'content': '{"context":"The user is asking for ways to improve both taste and nutritional value (specifically protein and vitamins) and explicitly asks for ways to increase vitamin content in their meal.","preference":"Values both flavor and nutritional content in meals and is interested in vitamin-rich food additions","categories":["food","nutrition","cooking","health"]}', 'memory_strategy_id': 'NutritionPreferences-mhcrH7Dq74', 'namespaces': ['/user-1/preferences']}, {'content': '{"context":"The user mentions they have a weightlifting competition coming up and are concerned about macros. Th

## Search User Preferences

Search the stored preferences to see what was captured.


In [41]:
# Search our user preferences namespace
search_namespace = (actor_id, "preferences")
result = store.search(search_namespace, query="food", limit=3)
print(f"Preferences namespace result: {result}")


Preferences namespace result: [Item(namespace=['user-1', 'preferences'], key='mem-1d32980c-8bf8-4a87-b071-78152aad4e3b', value={'content': '{"context":"The user mentions that salmon with rice and veggies is one of their favorite meals.","preference":"Enjoys salmon with rice and vegetables","categories":["food","meals","health"]}', 'memory_strategy_id': 'NutritionPreferences-mhcrH7Dq74', 'namespaces': ['/user-1/preferences']}, created_at='2026-01-03T18:57:51.737000+00:00', updated_at='2026-01-03T18:57:51.737000+00:00', score=0.3914623), Item(namespace=['user-1', 'preferences'], key='mem-fef53d3c-dd2a-4310-83d2-a48cfd809d5e', value={'content': '{"context":"The user explicitly refers to their meal as \'healthy\' and mentions the macros being great for their upcoming weightlifting competition. The user explicitly states they are focused on healthy eating with good macronutrient content.","preference":"Prefers healthy meals with good macronutrient profiles","categories":["food","health","nu

## Example 2: Second Conversation (New Session)

Run a second conversation in a new session. The agent should remember preferences from the previous conversation.


In [42]:
config = {
    "configurable": {
        "thread_id": "session-2", # New session ID
        "actor_id": actor_id, # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)


================================ Human Message =================================

Today's a new day, what should I make for dinner tonight?
pre_model_hook
Pre model hook messages: [HumanMessage(content="Today's a new day, what should I make for dinner tonight?", additional_kwargs={}, response_metadata={}, id='d99b61b0-3362-41fb-9f46-ab9e54948b01')]
================================== Ai Message ==================================

[User Context: {'content': '{"context":"The user mentions that salmon with rice and veggies is one of their favorite meals.","preference":"Enjoys salmon with rice and vegetables","categories":["food","meals","health"]}', 'memory_strategy_id': 'NutritionPreferences-mhcrH7Dq74', 'namespaces': ['/user-1/preferences']}, {'content': '{"context":"The user is asking for ways to improve both taste and nutritional value (specifically protein and vitamins) and explicitly asks for ways to increase vitamin content in their meal.","preference":"Values both flavor and nutrit